In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# Load dataset
df = pd.read_csv("Banglore_traffic_Dataset.csv")

print("Dataset loaded successfully ✅")
print("Shape of dataset:", df.shape)

# View first 5 rows
print(df.head())


Dataset loaded successfully ✅
Shape of dataset: (8936, 16)
         Date    Area Name Road/Intersection Name  Traffic Volume  \
0  2022-01-01  Indiranagar          100 Feet Road           50590   
1  2022-01-01  Indiranagar               CMH Road           30825   
2  2022-01-01   Whitefield    Marathahalli Bridge            7399   
3  2022-01-01  Koramangala    Sony World Junction           60874   
4  2022-01-01  Koramangala          Sarjapur Road           57292   

   Average Speed  Travel Time Index  Congestion Level  \
0      50.230299           1.500000        100.000000   
1      29.377125           1.500000        100.000000   
2      54.474398           1.039069         28.347994   
3      43.817610           1.500000        100.000000   
4      41.116763           1.500000        100.000000   

   Road Capacity Utilization  Incident Reports  Environmental Impact  \
0                 100.000000                 0               151.180   
1                 100.000000           

In [ ]:
print("\nDataset Info:")
print(df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8936 entries, 0 to 8935
Data columns (total 16 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Date                                8936 non-null   object 
 1   Area Name                           8936 non-null   object 
 2   Road/Intersection Name              8936 non-null   object 
 3   Traffic Volume                      8936 non-null   int64  
 4   Average Speed                       8936 non-null   float64
 5   Travel Time Index                   8936 non-null   float64
 6   Congestion Level                    8936 non-null   float64
 7   Road Capacity Utilization           8936 non-null   float64
 8   Incident Reports                    8936 non-null   int64  
 9   Environmental Impact                8936 non-null   float64
 10  Public Transport Usage              8936 non-null   float64
 11  Traffic Signal Compliance   

In [ ]:
print("\nMissing values in each column:")
print(df.isnull().sum())


Missing values in each column:
Date                                  0
Area Name                             0
Road/Intersection Name                0
Traffic Volume                        0
Average Speed                         0
Travel Time Index                     0
Congestion Level                      0
Road Capacity Utilization             0
Incident Reports                      0
Environmental Impact                  0
Public Transport Usage                0
Traffic Signal Compliance             0
Parking Usage                         0
Pedestrian and Cyclist Count          0
Weather Conditions                    0
Roadwork and Construction Activity    0
dtype: int64


In [ ]:
# Convert Date column
df['Date'] = pd.to_datetime(df['Date'])

# Sort data by time
df = df.sort_values(by='Date')

print("\nDate column converted and sorted ✔")



Date column converted and sorted ✔


In [ ]:
df.columns

Index(['Date', 'Area Name', 'Road/Intersection Name', 'Traffic Volume',
       'Average Speed', 'Travel Time Index', 'Congestion Level',
       'Road Capacity Utilization', 'Incident Reports', 'Environmental Impact',
       'Public Transport Usage', 'Traffic Signal Compliance', 'Parking Usage',
       'Pedestrian and Cyclist Count', 'Weather Conditions',
       'Roadwork and Construction Activity'],
      dtype='object')

In [ ]:
# Keep only useful columns
df = df[['Date', 'Traffic Volume']]

print("\nColumns after filtering:")
print(df.head())



Columns after filtering:
         Date  Traffic Volume
0  2022-01-01           50590
11 2022-01-01           15043
10 2022-01-01           38446
9  2022-01-01           31760
7  2022-01-01           25379


In [ ]:
# Create daily traffic series
df_daily = df.groupby('Date')['Traffic Volume'].mean().reset_index()

print("\nDaily aggregated traffic created ✔")
print(df_daily.head())



Daily aggregated traffic created ✔
        Date  Traffic Volume
0 2022-01-01    35587.666667
1 2022-01-02    27908.818182
2 2022-01-03    26660.428571
3 2022-01-04    29571.636364
4 2022-01-05    32738.500000


In [ ]:
df_daily.set_index('Date', inplace=True)

print("\nFinal cleaned dataset:")
print(df_daily.head())



Final cleaned dataset:
            Traffic Volume
Date                      
2022-01-01    35587.666667
2022-01-02    27908.818182
2022-01-03    26660.428571
2022-01-04    29571.636364
2022-01-05    32738.500000


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

In [ ]:
# initialize scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# fit and transform
scaled_data = scaler.fit_transform(df_daily[['Traffic Volume']])

print("\nScaling completed ✔")
print("First 5 scaled values:\n", scaled_data[:5])



Scaling completed ✔
First 5 scaled values:
 [[0.79851687]
 [0.50372644]
 [0.45580085]
 [0.56756191]
 [0.68913757]]


In [ ]:
joblib.dump(scaler, "traffic_scaler.pkl")
print("Scaler saved as traffic_scaler.pkl ✔")


Scaler saved as traffic_scaler.pkl ✔


In [ ]:
scaled_df = pd.DataFrame(scaled_data, index=df_daily.index, columns=['Traffic Volume'])

print("\nScaled dataset preview:")
print(scaled_df.head())



Scaled dataset preview:
            Traffic Volume
Date                      
2022-01-01        0.798517
2022-01-02        0.503726
2022-01-03        0.455801
2022-01-04        0.567562
2022-01-05        0.689138


In [ ]:
sequence_length = 30   # use past 30 days

X = []
y = []

for i in range(sequence_length, len(scaled_data)):
    X.append(scaled_data[i-sequence_length:i])
    y.append(scaled_data[i])


In [ ]:
X = np.array(X)
y = np.array(y)

print("Sequence creation completed ✔")
print("X shape:", X.shape)
print("y shape:", y.shape)


Sequence creation completed ✔
X shape: (922, 30, 1)
y shape: (922, 1)


In [ ]:
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)


Training samples: (737, 30, 1)
Testing samples: (185, 30, 1)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
model = Sequential()

# First LSTM layer
model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))

# Second LSTM layer
model.add(LSTM(64))
model.add(Dropout(0.2))

# Output layer
model.add(Dense(1))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

print(model.summary())


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,985 (195.25 KB)

 Trainable params: 49,985 (195.25 KB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.1602 - val_loss: 0.0301
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0262 - val_loss: 0.0253
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0293 - val_loss: 0.0253
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0248 - val_loss: 0.0268
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0270 - val_loss: 0.0255
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.0245 - val_loss: 0.0252
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - loss: 0.0235 - val_loss: 0.0265
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0238 - val_loss: 0.0254
Epoch 9/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0234 - val_loss: 0.0270
Epoch 10/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0260 - val_loss: 0.0284
Epoch 11/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0276 - val_loss: 0.0275
Epoch 12/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0

In [ ]:
predictions = model.predict(X_test)


6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 135ms/step


In [ ]:
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test)


In [ ]:
last_30_days = scaled_data[-30:]
last_30_days = last_30_days.reshape(1, 30, 1)

next_day_prediction = model.predict(last_30_days)
next_day_prediction = scaler.inverse_transform(next_day_prediction)

print("Predicted Next Day Traffic:", int(next_day_prediction[0][0]), "vehicles")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Predicted Next Day Traffic: 29845 vehicles


In [ ]:
model.save("traffic_lstm_model.h5")
print("Model saved ✔")


Model saved ✔


In [ ]:
joblib.dump(scaler, "traffic_scaler.pkl")
print("Scaler saved ✔")


Scaler saved ✔


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


In [ ]:
y_test_actual = scaler.inverse_transform(y_test)


In [ ]:
# MAE
mae = mean_absolute_error(y_test_actual, predictions)

# RMSE
rmse = np.sqrt(mean_squared_error(y_test_actual, predictions))

print("\nModel Evaluation 📊")
print("MAE:", round(mae, 2), "vehicles")
print("RMSE:", round(rmse, 2), "vehicles")



Model Evaluation 📊
MAE: 3271.03 vehicles
RMSE: 4161.81 vehicles


In [ ]:
def congestion_level(vehicle_count):

    if vehicle_count < 25000:
        return "LOW 🟢"

    elif vehicle_count < 40000:
        return "MEDIUM 🟡"

    else:
        return "HIGH 🔴"


In [ ]:
predicted_traffic = int(next_day_prediction[0][0])

level = congestion_level(predicted_traffic)

print("\nPredicted Traffic:", predicted_traffic, "vehicles")
print("Congestion Level:", level)



Predicted Traffic: 29845 vehicles
Congestion Level: MEDIUM 🟡
